# Week 6: Evaluating & Operationalizing Agents

**Project:** Signup Email Agent  
**Tools:** LangChain · LangSmith · DeepEval




*   https://smith.langchain.com/
*   https://platform.openai.com/
*   http://deepeval.com/





---
## Step 1: Create a Simple LangChain Agent


### Setup OpenAI and Langsmith

In [ ]:
%pip install -q langchain langchain_openai langchain_core langsmith deepeval pandas

In [ ]:
import os
from google.colab import userdata

# ─── OpenAI Keys ───
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ API keys set")


# ─── LangSmith Config ───
# To fix the 401 Unauthorized error, ensure you have saved your LangSmith API key in Colab's Secrets under 'LANGSMITH_API_KEY'
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY') or ""
os.environ["LANGSMITH_TRACING"] = "true"

print("✅ LangSmith tracing enabled")

✅ API keys set
✅ LangSmith tracing enabled


### Create the agent

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal


# Define the structured output
class SignupEmailOutput(BaseModel):
    to: str = Field(description="Email address")
    icp_fit: Literal["high", "medium", "low", "unknown"] = Field(description="ICP fit classification")
    reason: str = Field(description="Brief reasoning for classification")
    subject: str = Field(description="Email subject line")
    body: str = Field(description="Email body text")


# Text between the two """s is our prompt (FEEL FREE TO CHANGE)
SYSTEM_PROMPT = """
Process this signup to your dev tool SaaS called 'Glop'.
Return icp_fit, reason and welcome email.

Welcome email should mention user's current role and company.
Make sure the email is concise, conversationally informal, and professional.
Use relevant input fields when appropriate. Sound specific to the user amd avoid feeling like a generic template.
Do not mention details that are not supported by the input.
"""

# Create the agent
agent = create_agent(
    "openai:gpt-4o-mini",
    tools=[],  # No tools needed — this is a reasoning-only agent
    system_prompt=SYSTEM_PROMPT,
    response_format=SignupEmailOutput,
)

print("✅ Agent created with structured output")

✅ Agent created with structured output


### Test the agent with a single record

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Process this signup:
email: sarah@acmecorp.com
first_name: Sarah
company_name: Acme Corp
role: VP of Engineering
industry: SaaS
company_size: 500"""
    }]
})

# Extract structured response
output = result["structured_response"]
print(f"ICP Fit:  {output.icp_fit}")
print(f"Reason:   {output.reason}")
print(f"Subject:  {output.subject}")
print(f"Body:  {output.body}")

ICP Fit:  high
Reason:   The user's role as VP of Engineering in a SaaS company with 500 employees is a strong fit for Glop's offerings, targeting mid-sized to large tech firms.
Subject:  Welcome to Glop, Sarah!
Body:  Hi Sarah,

Welcome aboard! We're thrilled to have you at Glop. As the VP of Engineering at Acme Corp, your insights and experience in the SaaS industry will be incredibly valuable. 

If you have any questions or need assistance getting started, don’t hesitate to reach out.

Cheers,
The Glop Team


### Verify Traces in LangSmith

Open [smith.langchain.com](https://smith.langchain.com) → **default** project.

You should see:
- ✅ A trace for the single invocation above
- ✅ Full input/output captured
- ✅ Latency, token usage, and model info


---
## Step 2: Offline Evaluation with DeepEval




### Upload golden dataset to LangSmith (manually)

Before we start running the agent, we need a golden dataset in LangSmith. This is the ground truth we'll evaluate against later.

**Do this now in the LangSmith UI:**

0. Download the [golden_dataset.jsonl](https://drive.google.com/file/d/1PPpfHP5BIYGwReZ_8S7dK60vWWrAVrxb/view?usp=sharing) file
1. Go to [smith.langchain.com](https://smith.langchain.com)
2. Click **Datasets & Experiments** in the left sidebar
3. Click **+ Dataset**
4. Click **Click to upload**
5. Name it `golden-dataset`
6. Upload the [golden_dataset.jsonl](https://drive.google.com/file/d/1PPpfHP5BIYGwReZ_8S7dK60vWWrAVrxb/view?usp=sharing) file
7. Verify

### Fetch golden dataset locally

In [ ]:
from langsmith import Client

ls = Client()

dataset_name = "golden-dataset"
examples = list(ls.list_examples(dataset_name=dataset_name))

print(f"Loaded {len(examples)} examples")
print("Sample inputs keys:", examples[0].inputs.keys())
print("Sample outputs keys:", examples[0].outputs.keys())

Loaded 6 examples
Sample inputs keys: dict_keys(['messages'])
Sample outputs keys: dict_keys(['outputs_1'])


### Create evaluators with DeepEval

In [ ]:
from deepeval.metrics import JsonCorrectnessMetric, ExactMatchMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Exact match metric
icp_fit_evaluator = ExactMatchMetric(
    threshold=1.0,
)

# LLM-as-a-judge
email_evaluator = GEval(
    name="Email Quality",
    criteria="Evaluate if the email is professional, personalized (uses the person's name/role/company), concise (under 150 words), and does not hallucinate facts not present in the input.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.5
)

icp_fit_evaluator.include_reason = True
email_evaluator.include_reason = True

/tmp/ipykernel_2645/425009796.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


### Run evals across all golden dataset

In [ ]:
import json

icp_fit_scores = []
email_scores = []

for ex in examples:
    input = ex.inputs
    result = agent.invoke(input)

    actual = result["structured_response"]
    # Safely extract the expected output, accounting for potential 'outputs_1' nesting
    if "outputs_1" in ex.outputs:
        expected = ex.outputs["outputs_1"]
    else:
        # Fallback: assume ex.outputs directly contains the expected fields
        expected = ex.outputs

    icp_fit_tc = LLMTestCase(
        input=json.dumps(input),
        actual_output=actual.icp_fit,
        expected_output=expected.get("icp_fit", ""),
    )
    icp_fit_evaluator.measure(icp_fit_tc)
    icp_fit_scores.append(icp_fit_evaluator.score)


    email_tc = LLMTestCase(
        input=json.dumps(input),
        actual_output=actual.body,
        expected_output=expected.get("body", ""),
    )
    email_evaluator.measure(email_tc)
    email_scores.append(email_evaluator.score)
    print(email_evaluator.reason)

print(email_scores)

avg = sum(email_scores) / len(email_scores);

print("icp_fit exact match rate:", sum(icp_fit_scores) / len(icp_fit_scores))
print("email avg score:", sum(email_scores) / len(email_scores))

Output()

Output()

The email addresses the recipient by name, Rachel, and mentions her role and company, ensuring personalization. It is concise at 104 words, maintaining a professional tone throughout. However, it includes a reference to 'Glop' without context, which could be seen as a slight deviation from the input details, but overall, it aligns well with the evaluation steps.


Output()

Output()

...

Output()

The email addresses the recipient by their first name, which adds a level of personalization. However, it exceeds the word limit, containing more than 150 words, which detracts from its conciseness. The tone remains professional throughout, but the lack of specific information about the recipient's role or company could be seen as a shortcoming, as it does not fully utilize the personalization potential. Additionally, there are no hallucinated facts, aligning with the evaluation steps.


Output()

Output()

The email addresses the recipient by name, which adds a personal touch. However, it exceeds the word limit, containing 160 words instead of the recommended 150. The tone is professional throughout, but it includes a factual inaccuracy by stating 'join us at Glop' when the company is Finserv, indicating a lack of attention to detail in the content.
[0.883660620574962, 0.8768735751788151, 0.8506906338824338, 0.8850634726338054, 0.5664844997146525, 0.6615437882206496]
icp_fit exact match rate: 0.3333333333333333
email avg score: 0.787386098367553


---
## Step 3: Online Evaluation on LangSmith

Online eval scores **production traces** in real time. We'll set up rules in LangSmith that automatically evaluate every run.

### In the LangSmith UI:

1. Go to your **default** project
2. Click **Tracing** → **default** → **New** → **New Evaluator**
3. Create a Prebuilt Evaluator




### Run agent on a new input

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Process this signup:
        email: unknown@spam.com
        first_name: Whatever you want
        """
    }]
})

# Extract structured response
output = result["structured_response"]
print(f"ICP Fit:  {output.icp_fit}")
print(f"Reason:   {output.reason}")
print(f"Subject:  {output.subject}")

ICP Fit:  unknown
Reason:   Insufficient information to determine ICP fit with company and role details.
Subject:  Welcome to Glop!


### Check Online Evaluation Scores

Go back to [LangSmith](https://smith.langchain.com) → **default** project.

You should now see:
- ✅ A new trace from the last run
- ✅ **Feedback** attached to that trace


### What to look for:
1. Filter by feedback score to find failed cases
2. Click into failing traces to understand **why** they failed
3. Look at the **Monitor** tab for quality metrics

---
## Step 4: Add Online Eval Failures to the Golden Dataset

This is the production feedback loop: when online eval catches a bad output, you add that case to your golden dataset so offline eval catches it next time.

### In the LangSmith UI (recommended):

1. Find a trace that **failed** an online eval rule (or that you manually identify as wrong)
2. Click on the trace → **Add to Dataset**
3. Select **golden-dataset**
4. Fill in the **expected output**
5. Your golden dataset now has one more case — sourced from a real production failure

### The production feedback loop

```
┌─────────────────────────────────────────────────┐
│                                                 │
│   Agent runs on new signups                     │
│        │                                        │
│        ▼                                        │
│   LangSmith traces + online eval scores         │
│        │                                        │
│        ▼                                        │
│   Failures flagged automatically                │
│        │                                        │
│        ▼                                        │
│   Add failing cases to golden dataset           │
│        │                                        │
│        ▼                                        │
│   Re-run offline eval (DeepEval) with           │
│   expanded golden set                           │
│        │                                        │
│        ▼                                        │
│   Fix prompt / agent → redeploy                 │
│        │                                        │
│        └──────────── loop ──────────────────────┘
│                                                 │
└─────────────────────────────────────────────────┘
```

This is the core LLMOps pattern: **online eval catches issues → golden dataset grows → offline eval prevents regressions → agent improves.**